# Notebook Conclusion

In this notebook, the NYC Yellow Taxi trip dataset and Taxi Zone Lookup dataset were prepared for hotspot analysis.

The following preprocessing steps were completed:

1. Loaded the Yellow Taxi trip records and Taxi Zone metadata.
2. Selected pickup-related features relevant to hotspot detection.
3. Removed records with missing pickup information.
4. Validated pickup location identifiers against official NYC Taxi Zones.
5. Merged trip data with zone information to obtain meaningful geographic descriptions.
6. Engineered temporal features including pickup hour, day of week, month, and weekend indicators.
7. Performed final quality checks to ensure data integrity.
8. Saved the processed dataset for exploratory spatial analysis.

The prepared dataset is now ready for identifying and visualizing high-demand pickup hotspots across New York City.

---

## Final Dataset Structure

The processed dataset contains the following variables:

| Feature | Description |
|----------|-------------|
| tpep_pickup_datetime | Pickup timestamp |
| PULocationID | Pickup location identifier |
| LocationID | Taxi Zone identifier |
| Borough | NYC borough |
| Zone | Taxi pickup zone |
| service_zone | Taxi service area |
| pickup_hour | Hour of pickup |
| pickup_dayofweek | Day of pickup |
| pickup_month | Month of pickup |
| is_weekend | Weekend indicator |

# Use Case 3: Detect and Visualize High-Demand Pickup Hotspots

## Notebook 1: Data Preparation

### Problem Statement

Taxi service providers need to identify areas with consistently high pickup demand to improve driver allocation, reduce passenger waiting times, and optimize operational efficiency.

The objective of this use case is to detect and visualize high-demand pickup hotspots across New York City using taxi trip records.

### Objectives

- Prepare taxi trip data for hotspot analysis.
- Integrate Taxi Zone metadata for geographic interpretation.
- Create a clean dataset suitable for spatial demand analysis.
- Generate aggregated pickup demand statistics for each taxi zone.

### Dataset Information

#### 1. Yellow Taxi Trip Data (`yellow_tripdata_2026-01.parquet`)

Contains trip-level information including:

- Pickup timestamp
- Dropoff timestamp
- Pickup Location ID (`PULocationID`)
- Dropoff Location ID (`DOLocationID`)
- Fare information
- Passenger count

#### 2. Taxi Zone Lookup (`taxi_zone_lookup.csv`)

Maps Taxi Zone IDs to:

- Borough
- Zone Name
- Service Zone

This mapping enables interpretation of pickup hotspots using meaningful geographic locations.

---

## Import Libraries

Import the required libraries for data manipulation and analysis.

In [33]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


## Load Datasets

Load the Yellow Taxi trip dataset and Taxi Zone Lookup dataset.

In [34]:
# Load Yellow Taxi Dataset
trip_df = pd.read_parquet(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE3_Pickup_hotspot_Detection/data/yellow_tripdata_2026-01.parquet"
)

# Load Taxi Zone Lookup Dataset
zone_df = pd.read_csv(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE3_Pickup_hotspot_Detection/data/taxi_zone_lookup (2).csv"
)

print("Trip Dataset Shape:", trip_df.shape)
print("Zone Lookup Shape:", zone_df.shape)

Trip Dataset Shape: (3724889, 20)
Zone Lookup Shape: (265, 4)


## Trip Dataset Overview

Understand the structure and contents of the taxi trip dataset.

In [35]:
trip_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [36]:
trip_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [37]:
trip_df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
VendorID,3724889.0,NaN,NaN,NaN,1.873598,1.0,2.0,2.0,2.0,7.0,0.701458
tpep_pickup_datetime,3724889,NaN,NaN,NaN,2026-01-17 01:44:52.106518,2025-12-31 23:57:29,2026-01-09 17:51:59,2026-01-16 21:20:56,2026-01-24 07:25:11,2026-02-01 00:45:01,NaN
tpep_dropoff_datetime,3724889,NaN,NaN,NaN,2026-01-17 02:02:03.696138,2025-12-31 23:57:32,2026-01-09 18:08:47,2026-01-16 21:36:14,2026-01-24 07:40:22,2026-02-01 23:35:31,NaN
passenger_count,2636831.0,NaN,NaN,NaN,1.256271,0.0,1.0,1.0,1.0,9.0,0.670243
trip_distance,3724889.0,NaN,NaN,NaN,6.455647,0.0,1.0,1.81,3.73,269097.48,648.885528
RatecodeID,2636831.0,NaN,NaN,NaN,5.218853,1.0,1.0,1.0,1.0,99.0,19.653699
store_and_fwd_flag,2636831,2,N,2634494,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PULocationID,3724889.0,NaN,NaN,NaN,161.43719,1.0,114.0,161.0,233.0,265.0,67.066574
DOLocationID,3724889.0,NaN,NaN,NaN,160.99357,1.0,107.0,162.0,234.0,265.0,71.03604
payment_type,3724889.0,NaN,NaN,NaN,0.846564,0.0,0.0,1.0,1.0,4.0,0.712049


## Taxi Zone Lookup Overview

Examine the Taxi Zone mapping dataset.

In [38]:
zone_df.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [39]:
zone_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   LocationID    265 non-null    int64 
 1   Borough       264 non-null    object
 2   Zone          264 non-null    object
 3   service_zone  263 non-null    object
dtypes: int64(1), object(3)
memory usage: 8.4+ KB


In [40]:
zone_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
LocationID,265.0,NaN,NaN,NaN,133.0,76.643112,1.0,67.0,133.0,199.0,265.0
Borough,264,7,Queens,69,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zone,264,261,Governor's Island/Ellis Island/Liberty Island,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
service_zone,263,4,Boro Zone,205,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Feature Selection

Select features relevant for hotspot detection.

The following variables are required:

- `tpep_pickup_datetime` → Temporal analysis
- `PULocationID` → Pickup hotspot detection

In [41]:
pickup_df = trip_df[
    [
        "tpep_pickup_datetime",
        "PULocationID"
    ]
].copy()

pickup_df.head()

,tpep_pickup_datetime,PULocationID
0,2026-01-01 00:54:04,239
1,2026-01-01 00:34:04,163
2,2026-01-01 00:57:06,43
3,2026-01-01 00:15:22,142
4,2026-01-01 00:27:13,88


## Missing Value Analysis

Check for missing values in selected features.

In [42]:
pickup_df.isnull().sum()

tpep_pickup_datetime    0
PULocationID            0
dtype: int64

## Remove Missing Values

Rows with missing pickup timestamps or pickup location IDs cannot contribute to hotspot analysis.

In [43]:
initial_rows = len(pickup_df)

pickup_df.dropna(
    subset=[
        "tpep_pickup_datetime",
        "PULocationID"
    ],
    inplace=True
)

removed_rows = initial_rows - len(pickup_df)

print(f"Removed {removed_rows:,} rows.")
print(f"Remaining rows: {len(pickup_df):,}")

Removed 0 rows.
Remaining rows: 3,724,889


## Datetime Conversion

Convert pickup timestamps into datetime format to support temporal demand analysis.

In [44]:
pickup_df[
    "tpep_pickup_datetime"
] = pd.to_datetime(
    pickup_df[
        "tpep_pickup_datetime"
    ]
)

pickup_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 2 columns):
 #   Column                Dtype         
---  ------                -----         
 0   tpep_pickup_datetime  datetime64[us]
 1   PULocationID          int32         
dtypes: datetime64[us](1), int32(1)
memory usage: 42.6 MB


## Validate Pickup Location IDs

Ensure all pickup location IDs exist in the Taxi Zone Lookup table.

In [45]:
valid_locations = zone_df[
    "LocationID"
].unique()

pickup_df = pickup_df[
    pickup_df[
        "PULocationID"
    ].isin(
        valid_locations
    )
]

print(
    "Shape after validation:",
    pickup_df.shape
)

Shape after validation: (3724889, 2)


## Merge Taxi Zone Information

Join pickup records with zone metadata to obtain:

- Borough
- Zone Name
- Service Zone

This enables business interpretation of hotspot regions.

In [46]:
pickup_df = pickup_df.merge(
    zone_df,
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

In [48]:
pickup_df.shape
pickup_df.head()

,tpep_pickup_datetime,PULocationID,LocationID,Borough,Zone,service_zone
0,2026-01-01 00:54:04,239,239,Manhattan,Upper West Side South,Yellow Zone
1,2026-01-01 00:34:04,163,163,Manhattan,Midtown North,Yellow Zone
2,2026-01-01 00:57:06,43,43,Manhattan,Central Park,Yellow Zone
3,2026-01-01 00:15:22,142,142,Manhattan,Lincoln Square East,Yellow Zone
4,2026-01-01 00:27:13,88,88,Manhattan,Financial District South,Yellow Zone


## Verify Zone Mapping

Ensure every pickup location has a corresponding zone description.

In [49]:
pickup_df[
    [
        "Borough",
        "Zone"
    ]
].isnull().sum()

Borough    1539
Zone       4391
dtype: int64

## Temporal Feature Engineering

Extract time-related features from the pickup timestamp to support future temporal hotspot analysis.

The following features are generated:

- `pickup_hour`: Hour of pickup (0–23)
- `pickup_dayofweek`: Day name of pickup
- `pickup_month`: Month of pickup
- `is_weekend`: Indicates whether the trip occurred on a weekend

These features help analyze how pickup demand varies across different times and days.

In [50]:
# Pickup Hour
pickup_df["pickup_hour"] = (
    pickup_df["tpep_pickup_datetime"]
    .dt.hour
)

# Pickup Day Name
pickup_df["pickup_dayofweek"] = (
    pickup_df["tpep_pickup_datetime"]
    .dt.day_name()
)

# Pickup Month
pickup_df["pickup_month"] = (
    pickup_df["tpep_pickup_datetime"]
    .dt.month
)

# Weekend Indicator
pickup_df["is_weekend"] = (
    pickup_df["tpep_pickup_datetime"]
    .dt.weekday >= 5
).astype(int)

pickup_df.head()

,tpep_pickup_datetime,PULocationID,LocationID,Borough,Zone,service_zone,pickup_hour,pickup_dayofweek,pickup_month,is_weekend
0,2026-01-01 00:54:04,239,239,Manhattan,Upper West Side South,Yellow Zone,0,Thursday,1,0
1,2026-01-01 00:34:04,163,163,Manhattan,Midtown North,Yellow Zone,0,Thursday,1,0
2,2026-01-01 00:57:06,43,43,Manhattan,Central Park,Yellow Zone,0,Thursday,1,0
3,2026-01-01 00:15:22,142,142,Manhattan,Lincoln Square East,Yellow Zone,0,Thursday,1,0
4,2026-01-01 00:27:13,88,88,Manhattan,Financial District South,Yellow Zone,0,Thursday,1,0


## Verify Engineered Features

Inspect the newly created temporal features to ensure they have been generated correctly.

In [51]:
pickup_df[
    [
        "tpep_pickup_datetime",
        "pickup_hour",
        "pickup_dayofweek",
        "pickup_month",
        "is_weekend"
    ]
].head()

,tpep_pickup_datetime,pickup_hour,pickup_dayofweek,pickup_month,is_weekend
0,2026-01-01 00:54:04,0,Thursday,1,0
1,2026-01-01 00:34:04,0,Thursday,1,0
2,2026-01-01 00:57:06,0,Thursday,1,0
3,2026-01-01 00:15:22,0,Thursday,1,0
4,2026-01-01 00:27:13,0,Thursday,1,0


## Final Dataset Inspection

Review the structure and quality of the processed dataset before saving it for further analysis.

In [52]:
print("Dataset Shape:", pickup_df.shape)

Dataset Shape: (3724889, 10)


In [53]:
pickup_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 10 columns):
 #   Column                Dtype         
---  ------                -----         
 0   tpep_pickup_datetime  datetime64[us]
 1   PULocationID          int32         
 2   LocationID            int64         
 3   Borough               object        
 4   Zone                  object        
 5   service_zone          object        
 6   pickup_hour           int32         
 7   pickup_dayofweek      object        
 8   pickup_month          int32         
 9   is_weekend            int64         
dtypes: datetime64[us](1), int32(3), int64(2), object(4)
memory usage: 241.6+ MB


In [54]:
pickup_df.head()

,tpep_pickup_datetime,PULocationID,LocationID,Borough,Zone,service_zone,pickup_hour,pickup_dayofweek,pickup_month,is_weekend
0,2026-01-01 00:54:04,239,239,Manhattan,Upper West Side South,Yellow Zone,0,Thursday,1,0
1,2026-01-01 00:34:04,163,163,Manhattan,Midtown North,Yellow Zone,0,Thursday,1,0
2,2026-01-01 00:57:06,43,43,Manhattan,Central Park,Yellow Zone,0,Thursday,1,0
3,2026-01-01 00:15:22,142,142,Manhattan,Lincoln Square East,Yellow Zone,0,Thursday,1,0
4,2026-01-01 00:27:13,88,88,Manhattan,Financial District South,Yellow Zone,0,Thursday,1,0


In [59]:
pickup_df.shape

(3724889, 10)

In [55]:
pickup_df.isnull().sum()

tpep_pickup_datetime       0
PULocationID               0
LocationID                 0
Borough                 1539
Zone                    4391
service_zone            5930
pickup_hour                0
pickup_dayofweek           0
pickup_month               0
is_weekend                 0
dtype: int64

In [65]:
total_records = len(pickup_df)

missing_records = pickup_df["Zone"].isnull().sum()

missing_percentage = (
    missing_records / total_records
) * 100

print(f"Total Records: {total_records:,}")
print(f"Missing Zone Records: {missing_records:,}")
print(f"Percentage Missing: {missing_percentage:.4f}%")

Total Records: 3,724,889
Missing Zone Records: 4,391
Percentage Missing: 0.1179%


In [66]:
pickup_df = pickup_df.dropna(
    subset=[
        "Borough",
        "Zone",
        "service_zone"
    ]
)

print(
    pickup_df.isnull().sum()
)

tpep_pickup_datetime    0
PULocationID            0
LocationID              0
Borough                 0
Zone                    0
service_zone            0
pickup_hour             0
pickup_dayofweek        0
pickup_month            0
is_weekend              0
dtype: int64


## Handling Missing Zone Information

A small number of pickup records did not have corresponding entries in the Taxi Zone Lookup table.

Since these observations represented a negligible proportion of the dataset and could not be geographically interpreted, they were removed from further analysis.

This ensured that all pickup records used in hotspot analysis were associated with valid NYC taxi zones.

## Summary Statistics

Generate descriptive statistics for both numerical and categorical variables.

In [67]:
pickup_df.describe().T

,count,mean,min,25%,50%,75%,max,std
tpep_pickup_datetime,3718959,2026-01-17 01:46:12.836785,2025-12-31 23:57:29,2026-01-09 17:52:59,2026-01-16 21:22:13,2026-01-24 07:26:30.500000,2026-02-01 00:45:01,NaN
PULocationID,3718959.0,161.273237,1.0,114.0,161.0,233.0,263.0,66.99412
LocationID,3718959.0,161.273237,1.0,114.0,161.0,233.0,263.0,66.99412
pickup_hour,3718959.0,14.114554,0.0,10.0,15.0,19.0,23.0,5.985223
pickup_month,3718959.0,1.000018,1.0,1.0,1.0,1.0,12.0,0.013982
is_weekend,3718959.0,0.280748,0.0,0.0,0.0,1.0,1.0,0.449365


In [68]:
pickup_df.describe(
    include="object"
).T

,count,unique,top,freq
Borough,3718959,6,Manhattan,3180607
Zone,3718959,259,Upper East Side South,160343
service_zone,3718959,4,Yellow Zone,3047909
pickup_dayofweek,3718959,7,Saturday,671142


## Save Processed Dataset

Save the prepared dataset for subsequent exploratory analysis and hotspot detection.

The processed dataset contains spatial and temporal features required for identifying high-demand pickup hotspots.

In [69]:
pickup_df.to_parquet(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE3_Pickup_hotspot_Detection/data/processed_pickup_hotspot_data.parquet",
    index=False
)

print(
    "Processed dataset saved successfully."
)

Processed dataset saved successfully.


## Final Dataset Inspection

Review the final dataset structure before proceeding to exploratory analysis.

In [70]:
pickup_df.head()

,tpep_pickup_datetime,PULocationID,LocationID,Borough,Zone,service_zone,pickup_hour,pickup_dayofweek,pickup_month,is_weekend
0,2026-01-01 00:54:04,239,239,Manhattan,Upper West Side South,Yellow Zone,0,Thursday,1,0
1,2026-01-01 00:34:04,163,163,Manhattan,Midtown North,Yellow Zone,0,Thursday,1,0
2,2026-01-01 00:57:06,43,43,Manhattan,Central Park,Yellow Zone,0,Thursday,1,0
3,2026-01-01 00:15:22,142,142,Manhattan,Lincoln Square East,Yellow Zone,0,Thursday,1,0
4,2026-01-01 00:27:13,88,88,Manhattan,Financial District South,Yellow Zone,0,Thursday,1,0


In [71]:
pickup_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3718959 entries, 0 to 3724888
Data columns (total 10 columns):
 #   Column                Dtype         
---  ------                -----         
 0   tpep_pickup_datetime  datetime64[us]
 1   PULocationID          int32         
 2   LocationID            int64         
 3   Borough               object        
 4   Zone                  object        
 5   service_zone          object        
 6   pickup_hour           int32         
 7   pickup_dayofweek      object        
 8   pickup_month          int32         
 9   is_weekend            int64         
dtypes: datetime64[us](1), int32(3), int64(2), object(4)
memory usage: 269.5+ MB


In [72]:
pickup_df.isnull().sum()

tpep_pickup_datetime    0
PULocationID            0
LocationID              0
Borough                 0
Zone                    0
service_zone            0
pickup_hour             0
pickup_dayofweek        0
pickup_month            0
is_weekend              0
dtype: int64